In [4]:
!pip install numpy
!pip install jaxtyping


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /usr/local/fbcode/platform010/Python3.12.framework/Versions/3.12/bin/python3.12 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jaxtyping]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /usr/local/fbcode/platform010/Python3.12.framework/Versions/3.12/bin/python3.12 -m pip install --upgrade pip


In [17]:
import importlib
import tests.adapters
import torch
importlib.reload(tests.adapters)
from tests.adapters import run_cross_entropy
import torch.nn.functional as F
import numpy

In [22]:
    inputs = torch.tensor(
        [
            [
                [0.1088, 0.1060, 0.6683, 0.5131, 0.0645],
                [0.4538, 0.6852, 0.2520, 0.3792, 0.2675],
                [0.4578, 0.3357, 0.6384, 0.0481, 0.5612],
                [0.9639, 0.8864, 0.1585, 0.3038, 0.0350],
            ],
            [
                [0.3356, 0.9013, 0.7052, 0.8294, 0.8334],
                [0.6333, 0.4434, 0.1428, 0.5739, 0.3810],
                [0.9476, 0.5917, 0.7037, 0.2987, 0.6208],
                [0.8541, 0.1803, 0.2054, 0.4775, 0.8199],
            ],
        ]
    )
    print(inputs.shape)
    targets = torch.tensor([[1, 0, 2, 2], [4, 1, 4, 0]])

torch.Size([2, 4, 5])


In [25]:
y = numpy.arange(35).reshape(5, 7)

In [28]:
y[numpy.array([0, 2, 4]), numpy.array([0, 1, 2])]

array([ 0, 15, 30])

In [31]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
            state = self.state[p] # Get state associated with p.
            t = state.get("t", 0) # Get iteration number from the state, or 0.
            grad = p.grad.data # Get the gradient of loss with respect to p.
            p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
            state["t"] = t + 1 # Increment iteration number.
        return loss

In [46]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1)

In [73]:
def run_optimize(weights, opt):
    for t in range(10):
        opt.zero_grad() # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean() # Compute a scalar loss value.
        print(loss.cpu().item())
        loss.backward() # Run backward pass, which computes gradients.
        opt.step()

In [78]:
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt2 =  SGD([weights], lr=1e3)

In [79]:
run_optimize(weights, opt2)

26.84343719482422
9690.4814453125
1673698.5
186181104.0
15080668160.0
951762878464.0
48860395208704.0
2102183419969536.0
7.748199170388787e+16
2.488032910569898e+18


Problem (learning_rate_tuning): Tuning the learning rate (1 point)
As we will see, one of the hyperparameters that affects training the most is the learning rate.
Let’s see that in practice in our toy example. Run the SGD example above with three other values
for the learning rate: 1e1, 1e2, and 1e3, for just 10 training iterations. What happens with the loss
for each of these learning rates? Does it decay faster, slower, or does it diverge (i.e., increase over
the course of training)?

Loss with 1e1 decays but not to 0. 1e2 decays much faster to almost 0. 1e3 diverges

 

